In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import mysql.connector

In [2]:
stock = ["ICICIBANK.NS", "TCS.NS", "HCLTECH.NS", "SBIN.NS", "BHARTIARTL.NS", "LT.NS", "MARUTI.NS", "ASIANPAINT.NS", "TITAN.NS", "BAJFINANCE.NS"]
data = yf.download(stock, start = "2015-01-01", end = "2025-01-01")["Close"]
data.reset_index(inplace = True)
data.tail(5)

[*********************100%***********************]  10 of 10 completed


Ticker,Date,ASIANPAINT.NS,BAJFINANCE.NS,BHARTIARTL.NS,HCLTECH.NS,ICICIBANK.NS,LT.NS,MARUTI.NS,SBIN.NS,TCS.NS,TITAN.NS
2462,2024-12-24,2259.593262,676.644775,1570.768555,1814.316895,1287.317017,3606.118164,10621.637695,796.065369,3966.891846,3343.338135
2463,2024-12-26,2238.224121,677.395142,1586.239258,1817.951416,1287.416138,3595.962891,10779.331055,796.457520,3957.020996,3313.079346
2464,2024-12-27,2247.127930,686.538757,1586.586304,1809.582764,1297.538208,3574.760742,10823.898438,783.909485,3952.987305,3299.270996
2465,2024-12-30,2263.798096,684.600647,1573.743652,1844.922852,1284.736938,3545.879883,10683.320312,772.782837,3947.244629,3247.576660
2466,2024-12-31,2256.971924,678.115662,1574.586670,1833.876099,1271.836426,3574.314697,10742.133789,779.301941,3886.500244,3243.388916


In [3]:
conn = mysql.connector.connect(
    host = "127.0.0.1",
    user = "root",
    password = "@Mridul01",
    database = "portfolio_database"
)
cursor = conn.cursor()

In [4]:
for symbol in data.columns[1:]:

    for i,row in data.iterrows():

        cursor.execute(
        "INSERT INTO market_prices (symbol,date,close_price) VALUES (%s,%s,%s)",
        (symbol,row['Date'],row[symbol])
        )

conn.commit()

In [5]:
data = data.set_index("Date")

In [6]:
weights = {
    "ICICIBANK.NS" : 0.024,
    "TCS.NS" : 0.085,
    "HCLTECH.NS" : 0.113,
    "SBIN.NS" : 0.041,
    "BHARTIARTL.NS" : 0.204,
    "LT.NS" : 0.138,
    "MARUTI.NS" : 0.097,
    "ASIANPAINT.NS" : 0.103,
    "TITAN.NS" : 0.038,
    "BAJFINANCE.NS" : 0.157
}

weights = pd.Series(weights)
weights

ICICIBANK.NS     0.024
TCS.NS           0.085
HCLTECH.NS       0.113
SBIN.NS          0.041
BHARTIARTL.NS    0.204
LT.NS            0.138
MARUTI.NS        0.097
ASIANPAINT.NS    0.103
TITAN.NS         0.038
BAJFINANCE.NS    0.157
dtype: float64

In [7]:
returns = data.pct_change().dropna()
portfolio_returns = returns.dot(weights)
volatility = portfolio_returns.std()

In [8]:
var_95 = np.percentile(portfolio_returns.dropna(),5)
var_95

np.float64(-0.015927319031636578)

In [9]:
es_95 = portfolio_returns[portfolio_returns<= var_95].mean()

In [10]:
correlation_matrix = returns.corr()
rolling_volatility = portfolio_returns.rolling(window=30).std()
rolling_volatility

Date
2015-01-02         NaN
2015-01-05         NaN
2015-01-06         NaN
2015-01-07         NaN
2015-01-08         NaN
                ...   
2024-12-24    0.010356
2024-12-26    0.010238
2024-12-27    0.010032
2024-12-30    0.009937
2024-12-31    0.009939
Length: 2466, dtype: float64

In [11]:
annual_rf = 0.05
daily_rf = 0.05/252
sharpe_ratio = ((portfolio_returns.mean() - daily_rf)/volatility)*np.sqrt(252)
sharpe_ratio

np.float64(0.9397908351504429)

In [12]:
#Monte Carlo Simulation 
simulations = 1000
time_horizon = 252

simulated_returns = np.random.normal(portfolio_returns.mean(), 
                                     portfolio_returns.std(),
                                     (simulations, time_horizon)
)

sim = pd.DataFrame(simulated_returns)

sim["Simulation"] = sim.index

long_df = sim.melt(
    id_vars="Simulation",
    var_name="Day",
    value_name="Return"
)

In [13]:
initial_portfolio = 1000000

portfolio_paths = initial_portfolio * np.cumprod(1 + simulated_returns, axis=1)

port_path = pd.DataFrame(portfolio_paths)

port_path["Simulation"] = port_path.index

port_df = port_path.melt(
    id_vars="Simulation",
    var_name="Day",
    value_name="Portfolio_Value"
)


In [20]:
returns.to_csv("Returns.csv")
portfolio_returns.to_csv("Portfolio Returns.csv")
rolling_volatility.to_csv("Rolling Volatility.csv")


In [18]:
risk_metrics = {
    "Volatiltiy" : volatility,
    "VaR" : var_95,
    "Expected Shortfall" : es_95,
    "Sharpe Ratio" : sharpe_ratio,
}

pd.DataFrame([risk_metrics]).to_csv("Risk Metrics.csv", index = False)

In [19]:
long_df.to_csv("monte_carlo_powerbi.csv", index=False)
port_df.to_csv("portfolio_paths_powerbi.csv", index=False)